# Testing Django Views

## HTTP Requests and Responses
---

In this lesson, you'll learn how to **test Django views** - the HTTP layer of your application.

You have pytest-django set up. Now you'll test views, URL routing, and HTTP responses without starting a web server.

Examples use the **`blog`** app under `mysite/` (URLs live under the `/blog/` prefix). Run `pytest` from the directory that contains `manage.py`, as in lesson 15a.

1. [The Django Test Client](#The-Django-Test-Client),
    - [What Is the Test Client?](#What-Is-the-Test-Client?),
    - [The client Fixture](#The-client-Fixture),
2. [Testing GET Requests](#Testing-GET-Requests),
    - [Basic GET Request](#Basic-GET-Request),
    - [Response Attributes](#Response-Attributes),
    - [Checking Response Content](#Checking-Response-Content),
3. [Testing POST Requests](#Testing-POST-Requests),
    - [Submitting Form Data](#Submitting-Form-Data),
    - [Testing Redirects](#Testing-Redirects),
    - [CSRF Handling](#CSRF-Handling),
4. [URL Testing with reverse()](#URL-Testing-with-reverse()),
    - [Why Use reverse()](#Why-Use-reverse()),
    - [Testing URL Resolution](#Testing-URL-Resolution),
5. [Testing Template Context](#Testing-Template-Context),
    - [Accessing Context Variables](#Accessing-Context-Variables),
    - [Testing Template Used](#Testing-Template-Used),
6. [Authentication in Tests](#Authentication-in-Tests),
    - [Testing login-protected endpoints (Django admin)](#Testing-login-protected-endpoints-(Django-admin)),
    - [Using force_login](#Using-force_login),
7. [Practical Example: Testing blog views](#Practical-Example:-Testing-blog-views),
8. [🧠 EXERCISE 🧠, Test blog search (GET filters)](#🧠-EXERCISE-🧠,-Test-blog-search-(GET-filters)),
9. [🧠 EXERCISE 🧠, Test blog review form](#🧠-EXERCISE-🧠,-Test-blog-review-form).

<br>


## The Django Test Client

---

### What Is the Test Client?

---

The test client is a Python class that acts as a **dummy web browser**. It allows you to:

- Make GET, POST, PUT, DELETE requests
- Get responses without running a real server
- Test views, redirects, templates
- Handle cookies and sessions

<br>

```
┌─────────────────┐         ┌─────────────────┐
│   Test Client   │  ───▶   │   Django View   │
│   (fake browser)│         │                 │
│                 │  ◀───   │   Response      │
└─────────────────┘         └─────────────────┘
        │
        ▼
No real HTTP connection!
No server needed!
```

<br>

### The client Fixture

---

`pytest-django` provides a `client` fixture that gives you a test client instance:

In [ ]:
# Basic usage of the client fixture

def test_homepage(client):
    """Test that homepage returns 200."""
    response = client.get("/")
    
    assert response.status_code == 200

**Key point:** The `client` fixture is automatically provided by pytest-django. You don't need to create it or add database markers - it handles everything.

<br>

## Testing GET Requests

---

### Basic GET Request

---

Let's say we have a simple view:

In [ ]:
# blog/views.py (excerpt — same patterns as mysite/blog/views.py)

from django.http import HttpRequest, HttpResponse
from django.shortcuts import render
from django.views.decorators.http import require_http_methods

from blog.models import Blog


@require_http_methods(["GET"])
def blog_list(request: HttpRequest) -> HttpResponse:
    blogs = Blog.objects.all()
    return render(
        request,
        "blog/blog_list_static_example.html",
        {"blogs": blogs},
    )

# ...


In [ ]:
# blog/urls.py (excerpt — see mysite/blog/urls.py)

from django.urls import path
from blog import views

app_name = "blog"

urlpatterns = [
    path("", views.blog_list, name="blog_list"),
    path("<int:id>/", views.BlogDetailView.as_view(), name="blog_detail"),
    # ...
]


Test this view:

In [ ]:
# blog/tests/test_views.py

import pytest
from django.urls import reverse


@pytest.mark.django_db
def test_blog_list_returns_200(client):
    url = reverse("blog:blog_list")
    response = client.get(url)

    assert response.status_code == 200


<br>

### Response Attributes

---

The response object has many useful attributes:

| Attribute | Description | Example |
|-----------|-------------|---------|
| `status_code` | HTTP status code | `200`, `404`, `302` |
| `content` | Response body as bytes | `b'<html>...'` |
| `context` | Template context (if using templates) | `{'blogs': [...]}` |
| `templates` | List of templates used | `[<Template: blog/...html>]` |
| `headers` | Response headers | `{'Content-Type': 'text/html'}` |
| `url` | Final URL (after redirects) | `'/blog/'` |


In [ ]:
import pytest
from django.urls import reverse


@pytest.mark.django_db
def test_response_attributes(client):
    url = reverse("blog:blog_list")
    response = client.get(url)

    assert response.status_code == 200
    assert response["Content-Type"].startswith("text/html")

    template_names = [t.name for t in response.templates]
    assert "blog/blog_list_static_example.html" in template_names

    assert b"blog" in response.content.lower()


<br>

### Checking Response Content

---

You can check what's in the response body:

In [ ]:
import pytest
from datetime import date

from blog.models import Blog
from django.urls import reverse


@pytest.fixture
def blog(db):
    return Blog.objects.create(
        title="Django testing tips",
        author="janedoe",
        published_date=date(2026, 6, 1),
    )


@pytest.mark.django_db
def test_blog_list_shows_title(client, blog):
    url = reverse("blog:blog_list")
    response = client.get(url)

    assert blog.title.encode() in response.content
    assert blog.title in response.content.decode()


**Tip:** `response.content` is bytes, so either:
- Compare with bytes: `b"text" in response.content`
- Decode first: `"text" in response.content.decode()`

<br>

## Testing POST Requests

---

### Submitting Form Data

---

To test form submissions, use `client.post()` with a data dictionary:

In [ ]:
# blog/views.py (excerpt — blog_create in mysite)

from django.http import HttpRequest, HttpResponse
from django.shortcuts import render, redirect
from django.views.decorators.csrf import csrf_exempt
from django.views.decorators.http import require_http_methods

from blog.forms import BlogModelForm


@csrf_exempt
@require_http_methods(["GET", "POST"])
def blog_create(request: HttpRequest) -> HttpResponse:
    form = BlogModelForm(request.POST or None)
    if form.is_valid():
        blog = form.save(commit=False)
        if not blog.slug:
            blog.slug = blog.title.lower().replace(" ", "-")
        blog.save()
        return redirect("blog:blog_detail", id=blog.id)
    return render(request, "blog/blog_form.html", {"form": form})


In [ ]:
# blog/tests/test_views.py

import pytest
from django.urls import reverse

from blog.models import Blog


@pytest.mark.django_db
def test_blog_create_post(client):
    url = reverse("blog:blog_create")
    data = {
        "title": "My new blog",
        "author": "alice",
        "author_email": "",
        "content": "one two three four five",
        "published_date": "2026-03-01",
        "category_type": "1",
    }
    response = client.post(url, data)

    assert response.status_code == 302
    assert Blog.objects.filter(title="My new blog").exists()


<br>

### Testing Redirects

---

When a view redirects, you can check the redirect URL:

In [ ]:
import pytest
from django.urls import reverse
from blog.models import Blog


@pytest.mark.django_db
def test_blog_create_redirects_to_detail(client):
    url = reverse("blog:blog_create")
    data = {
        "title": "Redirect me",
        "author": "bob",
        "author_email": "",
        "content": "alpha beta gamma delta",
        "published_date": "2026-02-01",
        "category_type": "1",
    }
    response = client.post(url, data)

    assert response.status_code == 302
    blog = Blog.objects.get(title="Redirect me")
    expected = reverse("blog:blog_detail", kwargs={"id": blog.id})
    assert response.url == expected


**Follow the redirect:**

In [ ]:
import pytest
from django.urls import reverse


@pytest.mark.django_db
def test_blog_create_follow_redirect(client):
    url = reverse("blog:blog_create")
    data = {
        "title": "Follow redirect",
        "author": "carol",
        "author_email": "",
        "content": "one two three four words",
        "published_date": "2026-02-10",
        "category_type": "1",
    }
    response = client.post(url, data, follow=True)

    assert response.status_code == 200
    assert "Follow redirect".encode() in response.content


<br>

### CSRF Handling

---

Django’s **`Client()`** (and pytest-django’s **`client`** fixture) default to **`enforce_csrf_checks=False`**: unsafe requests **do not** require a CSRF token, so a bare `post()` often returns **200** — unlike a real browser session.

To exercise the same rule as production, build the client with **`Client(enforce_csrf_checks=True)`**. Then a POST **without** `csrfmiddlewaretoken` should get **403 Forbidden** (assuming the view is **not** `@csrf_exempt`).

In [ ]:
from django.test import Client

import pytest
from django.urls import reverse


@pytest.mark.django_db
def test_csrf_enforced_on_comment_create():
    url = reverse("blog:comment_create")
    data = {"content": "one two three four"}

    # No token: default Client() still reaches the view (often 200).
    assert Client().post(url, data).status_code == 200

    # No token + enforce_csrf_checks=True → 403 (comment_create is not @csrf_exempt).
    strict = Client(enforce_csrf_checks=True)
    assert strict.post(url, data).status_code == 403


<br>

## URL Testing with `reverse()`

---

### Why Use `reverse()`

---

**Don't hardcode URLs in tests!**

In [ ]:
import pytest
from django.urls import reverse


@pytest.mark.django_db
def test_blog_list_bad(client):
    # Hardcoded URL — breaks if you change the path() prefix in mysite/urls.py
    response = client.get("/blog/")
    assert response.status_code == 200


@pytest.mark.django_db
def test_blog_list_good(client):
    url = reverse("blog:blog_list")
    response = client.get(url)
    assert response.status_code == 200


**Benefits of `reverse()`:**
- Tests don't break when URLs change
- Catches URL configuration errors
- Documents which view is being tested

<br>

**URLs with parameters:**

In [ ]:
import pytest
from datetime import date
from django.urls import reverse

from blog.models import Blog


@pytest.fixture
def blog(db):
    return Blog.objects.create(
        title="Detail title",
        author="janedoe",
        published_date=date(2026, 5, 1),
    )


@pytest.mark.django_db
def test_blog_detail(client, blog):
    # URL kwarg is `id`, not pk (see blog/urls.py).
    url = reverse("blog:blog_detail", kwargs={"id": blog.pk})
    response = client.get(url)
    assert response.status_code == 200
    assert blog.title in response.content.decode()


<br>

### Testing URL Resolution

---

You can also test that URLs resolve to the correct views:

In [ ]:
# blog/tests/test_urls.py

from django.urls import reverse, resolve

from blog import views


def test_blog_list_url_resolves():
    url = reverse("blog:blog_list")
    resolver = resolve(url)
    assert resolver.func == views.blog_list


def test_blog_detail_url_resolves():
    url = reverse("blog:blog_detail", kwargs={"id": 42})
    resolver = resolve(url)
    assert resolver.func.view_class == views.BlogDetailView
    assert resolver.kwargs["id"] == 42


<br>

## Testing Template Context

---

### Accessing Context Variables

---

When a view uses `render()`, you can access the template context:

In [ ]:
import pytest
from datetime import date
from django.urls import reverse

from blog.models import Blog


@pytest.fixture
def blog(db):
    return Blog.objects.create(
        title="Context blog",
        author="janedoe",
        published_date=date(2026, 4, 1),
    )


@pytest.mark.django_db
def test_blog_list_context(client, blog):
    url = reverse("blog:blog_list")
    response = client.get(url)

    assert "blogs" in response.context
    blogs = list(response.context["blogs"])
    assert len(blogs) == 1
    assert blogs[0] == blog


In [ ]:
import pytest
from datetime import date
from django.urls import reverse

from blog.models import Blog


@pytest.fixture
def blog(db):
    return Blog.objects.create(
        title="Refactoring",
        author="martinfowler",
        published_date=date(2026, 1, 15),
    )


@pytest.mark.django_db
def test_blog_detail_context(client, blog):
    url = reverse("blog:blog_detail", kwargs={"id": blog.pk})
    response = client.get(url)

    assert response.context["blog"] == blog
    assert response.context["blog"].title == blog.title


<br>

### Testing Template Used

---

In [ ]:
import pytest
from django.urls import reverse


@pytest.mark.django_db
def test_blog_list_uses_correct_template(client):
    url = reverse("blog:blog_list")
    response = client.get(url)
    template_names = [t.name for t in response.templates]
    assert "blog/blog_list_static_example.html" in template_names


<br>

## Authentication in Tests

---

### Testing login-protected endpoints (Django admin)

---

For endpoints that require authentication, the notebook uses **Django admin** — it is always present and login-protected (URL name **`admin:index`**, resolved with **`reverse()`**), unlike the course `blog` views which are mostly open:

In [ ]:
# Django admin is login-protected in every project (mysite has admin enabled).
# urlpatterns: path("admin/", admin.site.urls)  → URL name "admin:index"
# Anonymous GET reverse("admin:index") → redirect to the login page.


In [ ]:
import pytest
from django.urls import reverse


@pytest.mark.django_db
def test_admin_requires_login(client):
    response = client.get(reverse("admin:index"))
    assert response.status_code == 302
    assert "login" in response.url


<br>

### Using `force_login`

---

To test as an authenticated user, use `client.force_login()`:

In [ ]:
# blog/tests/conftest.py

import pytest
from django.contrib.auth import get_user_model

User = get_user_model()


@pytest.fixture
def user(db):
    return User.objects.create_user(
        username="testuser",
        email="test@example.com",
        password="testpass123",
    )


@pytest.fixture
def admin_user(db):
    return User.objects.create_superuser(
        username="admin",
        email="admin@example.com",
        password="adminpass123",
    )


In [ ]:
# blog/tests/test_views.py

import pytest
from django.urls import reverse


@pytest.mark.django_db
def test_admin_index_for_superuser(client, admin_user):
    client.force_login(admin_user)
    response = client.get(reverse("admin:index"))
    assert response.status_code == 200


**Note:** `force_login()` skips password verification - faster than `login()` with credentials.

This test does not verify the login form process; it solely verifies that an already authenticated user is successfully granted access to the admin panel.

<br>

## Practical Example: Testing blog views

---

Let's write comprehensive tests for the **blog** app (`mysite/blog`) — the same URLs and templates you use with `runserver`.

<br>

**Working directory:** run `pytest` from the directory that contains `manage.py` (in this repo: `mysite/`), matching lesson 15a.


In [ ]:
# blog/views.py (composite excerpt — see mysite/blog/views.py)

from django.http import HttpRequest, HttpResponse
from django.shortcuts import render, redirect, get_object_or_404
from django.views import View
from django.views.decorators.csrf import csrf_exempt
from django.views.decorators.http import require_http_methods

from blog.models import Blog
from blog.forms import BlogModelForm


@require_http_methods(["GET"])
def blog_list(request: HttpRequest) -> HttpResponse:
    blogs = Blog.objects.all()
    return render(request, "blog/blog_list_static_example.html", {"blogs": blogs})


class BlogDetailView(View):
    def get(self, request: HttpRequest, id: int) -> HttpResponse:
        blog = get_object_or_404(Blog, id=id)
        return render(request, "blog/detail_preview.html", {"blog": blog})


@csrf_exempt
@require_http_methods(["GET", "POST"])
def blog_create(request: HttpRequest) -> HttpResponse:
    form = BlogModelForm(request.POST or None)
    if form.is_valid():
        blog = form.save(commit=False)
        if not blog.slug:
            blog.slug = blog.title.lower().replace(" ", "-")
        blog.save()
        return redirect("blog:blog_detail", id=blog.id)
    return render(request, "blog/blog_form.html", {"form": form})


In [ ]:
# blog/tests/test_views.py

import pytest
from datetime import date
from django.urls import reverse

from blog.models import Blog


@pytest.fixture
def blog(db):
    return Blog.objects.create(
        title="Refactoring",
        author="martinfowler",
        published_date=date(2026, 1, 15),
    )


class TestBlogListView:
    def test_returns_200(self, client, db):
        url = reverse("blog:blog_list")
        assert client.get(url).status_code == 200

    def test_uses_correct_template(self, client, db):
        url = reverse("blog:blog_list")
        response = client.get(url)
        names = [t.name for t in response.templates]
        assert "blog/blog_list_static_example.html" in names

    def test_shows_blog_title(self, client, blog):
        url = reverse("blog:blog_list")
        response = client.get(url)
        assert blog.title in response.content.decode()

    def test_empty_list(self, client, db):
        url = reverse("blog:blog_list")
        response = client.get(url)
        assert response.context["blogs"].count() == 0


class TestBlogDetailView:
    def test_returns_200(self, client, blog):
        url = reverse("blog:blog_detail", kwargs={"id": blog.pk})
        assert client.get(url).status_code == 200

    def test_returns_404(self, client, db):
        url = reverse("blog:blog_detail", kwargs={"id": 99999})
        assert client.get(url).status_code == 404

    def test_blog_in_context(self, client, blog):
        url = reverse("blog:blog_detail", kwargs={"id": blog.pk})
        response = client.get(url)
        assert response.context["blog"] == blog


class TestBlogCreateView:
    def test_get_returns_200(self, client, db):
        url = reverse("blog:blog_create")
        assert client.get(url).status_code == 200

    def test_post_valid_redirects(self, client, db):
        url = reverse("blog:blog_create")
        data = {
            "title": "Created via test",
            "author": "diana",
            "author_email": "",
            "content": "one two three four five",
            "published_date": "2026-04-01",
            "category_type": "1",
        }
        response = client.post(url, data)
        assert response.status_code == 302
        assert Blog.objects.filter(title="Created via test").exists()

    def test_form_in_context_on_get(self, client, db):
        url = reverse("blog:blog_create")
        response = client.get(url)
        assert "form" in response.context


<br>

## Summary

---

| Concept | Description | Example |
|---------|-------------|---------|
| **client fixture** | Test client for HTTP requests | `client.get(url)` |
| **client.get()** | Send GET request | `response = client.get("/")` |
| **client.post()** | Send POST request | `client.post(url, data)` |
| **response.status_code** | HTTP status code | `assert response.status_code == 200` |
| **response.context** | Template context | `response.context["blogs"]` |
| **reverse()** | Get URL from name | `reverse("blog:blog_list")` |
| **force_login()** | Authenticate user | `client.force_login(user)` |
| **follow=True** | Follow redirects | `client.post(url, data, follow=True)` |

<br>

**Key principles:**

1. **Use `reverse()`** — avoid hardcoding `/blog/...`
2. **Test status codes** — 200, 302, 404, 403
3. **Check context** — `blogs` vs `blog` depends on the view
4. **Test authentication** — e.g. `reverse("admin:index")` vs anonymous users
5. **Test redirects** — compare `response.url` to `reverse(...)`


<br>

### 🧠 EXERCISE 🧠, Test a Book List View

---

Write tests for **`blog_search`** — the real view filters by optional query `q` (title contains) and sorts via `sort` (see `mysite/blog/views.py` and `blog/search.html`).

<br>

**View (excerpt):**

```python
from django.http import HttpRequest, HttpResponse
from django.shortcuts import render
from django.views.decorators.http import require_http_methods

from blog.forms import BlogSearchForm
from blog.models import Blog


@require_http_methods(["GET"])
def blog_search(request: HttpRequest) -> HttpResponse:
    form = BlogSearchForm(request.GET)
    blogs = []
    if form.is_valid():
        query = form.cleaned_data["q"]
        sort = form.cleaned_data["sort"]
        blogs = Blog.objects.all()
        if query:
            blogs = blogs.filter(title__icontains=query)
        blogs = blogs.order_by(sort)
    return render(request, "blog/search.html", {"form": form, "blogs": blogs})
```

<br>

**Your task**

1. GET without filters returns **200**.
2. With two `Blog` rows in the DB, an empty search shows **both** titles in the response body.
3. `?q=<substring>` returns only blogs whose **title** matches (case-insensitive).
4. Response context contains **`form`** and **`blogs`**.


<details>
    <summary>▶️ Solution</summary>

```python
# blog/tests/test_views.py

import pytest
from datetime import date
from django.urls import reverse

from blog.models import Blog


@pytest.fixture
def blog_alpha(db):
    return Blog.objects.create(
        title="Django ORM tips",
        author="alice",
        published_date=date(2026, 1, 10),
    )


@pytest.fixture
def blog_beta(db):
    return Blog.objects.create(
        title="Python testing guide",
        author="bob",
        published_date=date(2026, 2, 20),
    )


@pytest.mark.django_db
class TestBlogSearchView:
    def test_returns_200(self, client, db):
        url = reverse("blog:blog_search")
        assert client.get(url).status_code == 200

    def test_shows_all_when_no_query(self, client, blog_alpha, blog_beta):
        url = reverse("blog:blog_search")
        response = client.get(url)
        body = response.content.decode()
        assert blog_alpha.title in body
        assert blog_beta.title in body

    def test_filters_by_title(self, client, blog_alpha, blog_beta):
        url = reverse("blog:blog_search")
        response = client.get(url, {"q": "django"})
        blogs = list(response.context["blogs"])
        assert len(blogs) == 1
        assert blogs[0] == blog_alpha

    def test_context_keys(self, client, db):
        url = reverse("blog:blog_search")
        response = client.get(url)
        assert "form" in response.context
        assert "blogs" in response.context
```

</details>


<br>

### 🧠 EXERCISE 🧠, Test Form Submission

---

Write tests for **`review_create`** (`/blog/review-form/`) using `BlogReviewForm` — see `mysite/blog/views.py` and `blog/forms.py`.

<br>

**Behaviour**

- GET returns **200** and includes **`form`** in context.
- Valid POST creates a **`BlogReview`**. The demo view hard-codes `user_id=1`; the solution fixture clears users so the created user gets **pk=1**.
- Successful POST **redirects** to `blog:review_form_success`.
- Invalid POST (e.g. missing **`rating`**) returns **200** and field errors on the form.

<br>

**Your task:** cover the four behaviours above.


<details>
    <summary>▶️ Solution</summary>

```python
# blog/tests/test_views.py

import pytest
from datetime import date
from django.contrib.auth import get_user_model
from django.urls import reverse

from blog.models import Blog, BlogReview

User = get_user_model()


@pytest.fixture
def user_pk1_for_review(db):
    User.objects.all().delete()
    u = User.objects.create_user(
        username="reviewer",
        email="r@example.com",
        password="secret123",
    )
    assert u.pk == 1  # review_create uses user_id=1
    return u


@pytest.fixture
def blog(db):
    return Blog.objects.create(
        title="Post for reviews",
        author="eve",
        published_date=date(2026, 3, 1),
    )


@pytest.mark.django_db
class TestReviewCreateView:
    def test_get_shows_form(self, client, blog, user_pk1_for_review):
        url = reverse("blog:review_form")
        response = client.get(url)
        assert response.status_code == 200
        assert "form" in response.context

    def test_valid_post_creates_review(self, client, blog, user_pk1_for_review):
        url = reverse("blog:review_form")
        data = {"blog": str(blog.pk), "rating": "5", "comment": "Great read."}
        response = client.post(url, data)
        assert response.status_code == 302
        assert BlogReview.objects.filter(blog=blog, rating=5).exists()

    def test_valid_post_redirects(self, client, blog, user_pk1_for_review):
        url = reverse("blog:review_form")
        data = {"blog": str(blog.pk), "rating": "4", "comment": "ok"}
        response = client.post(url, data)
        assert response.url == reverse("blog:review_form_success")

    def test_invalid_post_missing_rating(self, client, blog, user_pk1_for_review):
        url = reverse("blog:review_form")
        data = {"blog": str(blog.pk), "comment": "no rating"}
        response = client.post(url, data)
        assert response.status_code == 200
        assert "rating" in response.context["form"].errors
        assert not BlogReview.objects.filter(blog=blog, comment="no rating").exists()
```

</details>


---